# Bagian 3 — Varian 2 — Penentuan k dengan Silhouette Score

**Proyek Akhir Pembelajaran Mesin (INF62325) · Cabang C — Klastering K-Means**
**S1 Informatika · UIN Siber Syekh Nurjati Cirebon**

| | |
|---|---|
| Pertemuan | dikerjakan setelah Pertemuan 12, dipresentasikan Pertemuan 14 |
| Tahap resmi | Tahap 2 — Varian Eksperimen 2 |
| Dikerjakan oleh | **Anggota 2** saja |
| Berkas yang dibutuhkan | `data_bersih.csv` dari Bagian 1 saja |
| Luaran wajib | `data/hasil_varian/varian2.json`, `gambar/varian2_silhouette.png` |
| Dinilai pada | Progres 2, Pertemuan 14, **35% perorangan** |
| Tenggat | selesai sebelum Pertemuan 14 |

| | |
|---|---|
| Kelompok | Kelompok 4 (Kelas 5B) |
| Paket dataset | 5B-4 — IBM HR Employee Attrition |
| Nama dan NIM | Faqih Huddin SM. (2488010061) — Anggota 2, Varian Silhouette |

---

## Panduan Bagian 3

> **Ketentuan Varian 2:** *Silhouette Score untuk tiap k. Jika kesimpulannya berbeda dari Elbow, jelaskan mengapa.*

Bagian ini dikerjakan **Anggota 2 saja**, dan dinilai **perorangan**.

**Urutan kerja**

1. Jalankan 3.0 (persiapan).
2. Jalankan 3.1. Anda hanya membutuhkan `data_bersih.csv`. Tidak perlu menunggu anggota lain.
3. Jalankan 3.2.
4. **Tulis sendiri** jawaban di 3.3.
5. Jalankan 3.4, commit dengan akun Anda sendiri, lalu kirim `varian2.json` ke Anggota 3.

**Yang perlu dipahami lebih dulu**

Silhouette Score mengukur, untuk setiap titik, seberapa dekat ia ke klasternya sendiri dibanding ke klaster terdekat berikutnya. Nilainya antara −1 dan 1. Berbeda dari inertia, nilainya **tidak** otomatis membaik ketika k dinaikkan, jadi nilai tertinggi memang dapat dipakai sebagai pertimbangan.

Patokan tafsiran yang dipakai mata kuliah ini:

| Nilai | Tafsiran |
|---|---|
| di atas 0,50 | pemisahan klaster jelas |
| 0,25 sampai 0,50 | struktur lemah, tetapi masih dapat dipertanggungjawabkan |
| di bawah 0,25 | struktur hampir tidak ada |

Pada data pelanggan nyata, angka 0,20 sampai 0,35 adalah hal biasa. Angka rendah bukan berarti pekerjaan Anda salah; yang salah adalah melaporkannya seolah-olah pemisahannya jelas.

**Satu jebakan yang sering terjadi.** Silhouette hampir selalu tertinggi di k=2, karena membagi data menjadi dua bagian besar paling mudah dipisahkan. Nilai itu biasanya tidak dipakai, sebab dua segmen tidak memberi informasi yang berguna untuk tindakan. Kalau k Anda bukan nilai dengan Silhouette tertinggi, **itu justru perlu dijelaskan**, bukan disembunyikan.

Bila kesimpulan Anda berbeda dari Bagian 2, sel di bawah menyusun kerangka penjelasannya secara otomatis — tetapi Anda tetap harus menuliskannya dengan kalimat Anda sendiri.

---

## 3.0 Persiapan — **jalankan setiap kali notebook dibuka**

Sesi Colab menghapus seluruh berkasnya setiap kali runtime dimulai ulang. Karena itu hasil kerja tidak disimpan di sesi Colab, melainkan di **satu folder Google Drive** yang dipakai bersama sepanjang Pertemuan 12 sampai 15. Empat sel di bawah menghubungkan folder itu, memasang pustaka, menulis `alat.py`, dan menyiapkan fungsi pembantu. Keempatnya dijalankan lebih dulu, apa pun yang Anda kerjakan di notebook ini.

`alat.py` berisi fungsi yang dipakai ketiga varian. **Jangan mengubahnya sendiri-sendiri:** kalau satu anggota mengubah isi berkas ini, ketiga varian berhenti sebanding dan tabel perbandingan menjadi tidak sah.

In [ ]:
# Folder Drive ini adalah tempat serah-terima antar pertemuan: apa pun yang
# ditulis di sini terbaca kembali oleh notebook pertemuan berikutnya.
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

FOLDER = '/content/drive/.shortcut-targets-by-id/1kVQHXVw8QsRBgDDIfabawCG8Kc5AWXBQ/ProyekML_KlasteringKMeans'
os.makedirs(FOLDER, exist_ok=True)
os.chdir(FOLDER)      # sejak baris ini, 'data/...' berarti di dalam FOLDER

if FOLDER not in sys.path:
    sys.path.insert(0, FOLDER)   # agar 'import alat' menemukan berkas di folder ini

for d in ('data', 'data/hasil_varian', 'model', 'gambar'):
    os.makedirs(d, exist_ok=True)

print('Folder proyek:', FOLDER)
print('Isi folder   :', sorted(os.listdir()) or '(masih kosong)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder proyek: /content/drive/.shortcut-targets-by-id/1kVQHXVw8QsRBgDDIfabawCG8Kc5AWXBQ/ProyekML_KlasteringKMeans
Isi folder   : ['__pycache__', 'alat.py', 'data', 'gambar', 'model']


In [ ]:
import ast, json, os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

SEED = 42
ID_COL = 'EmployeeNumber'   # kolom identitas dataset IBM HR (sama dengan notebook 01)
IQR_PENGALI = 1.5

print('Siap. scikit-learn', sklearn.__version__)
print('Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.')

Siap. scikit-learn 1.6.1
Catat versi ini. Angka yang sama nanti ditulis di requirements.txt.


In [ ]:
%%writefile alat.py
"""alat.py -- fungsi bersama ketiga varian. Jangan diubah sendiri-sendiri."""
from __future__ import annotations

import json
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (calinski_harabasz_score,
                             davies_bouldin_score, silhouette_score)
from sklearn.preprocessing import StandardScaler

SEED = 42
ID_COL = 'EmployeeNumber'
RENTANG_K = range(2, 11)
BERKAS_BERSIH = 'data/data_bersih.csv'
FOLDER_HASIL = 'data/hasil_varian'

# Label dan satuan untuk antarmuka aplikasi (ketentuan Tahap 3: setiap
# masukan harus punya label dan satuan yang dipahami orang luar).
LABEL = {
    'Age': ('Usia karyawan', 'tahun'),
    'MonthlyIncome': ('Pendapatan bulanan', 'satuan moneter dataset'),
    'TotalWorkingYears': ('Total pengalaman kerja', 'tahun'),
    'YearsAtCompany': ('Lama bekerja di perusahaan', 'tahun'),
    'YearsSinceLastPromotion': ('Lama sejak promosi terakhir', 'tahun'),
    'NumCompaniesWorked': ('Jumlah perusahaan tempat bekerja', 'perusahaan'),
    'DistanceFromHome': ('Jarak rumah ke kantor', 'satuan jarak dataset'),
    'JobSatisfaction': ('Kepuasan kerja', 'skor 1-4'),
    'EnvironmentSatisfaction': ('Kepuasan lingkungan kerja', 'skor 1-4'),
    'WorkLifeBalance': ('Keseimbangan kerja-kehidupan', 'skor 1-4'),
    'JobInvolvement': ('Keterlibatan kerja', 'skor 1-4'),
    'OverTime_Yes': ('Status lembur (0=Tidak, 1=Ya)', '0 atau 1'),
}


def label_satuan(kolom: str) -> tuple[str, str]:
    return LABEL.get(kolom, (kolom.replace('_', ' ').title(), ''))


def muat_bersih(fitur: list[str] | None = None) -> pd.DataFrame:
    """Membaca data bersih kelompok. Kolom identitas selalu dibuang."""
    df = pd.read_csv(BERKAS_BERSIH)
    # Pengaman ganda: buang kolom identitas secara eksplisit meskipun bertipe numerik.
    df = df.drop(columns=[ID_COL], errors='ignore')
    df = df.select_dtypes(include='number')
    return df[fitur] if fitur else df


def skalakan(df: pd.DataFrame):
    sc = StandardScaler()
    return sc.fit_transform(df), sc


def jalankan_k(X_scaled, rentang=RENTANG_K) -> pd.DataFrame:
    """Menjalankan K-Means untuk tiap k, mengembalikan tabel metrik."""
    baris = []
    for k in rentang:
        km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    random_state=SEED)
        lab = km.fit_predict(X_scaled)
        baris.append({
            'k': k,
            'inertia': float(km.inertia_),
            'silhouette': float(silhouette_score(X_scaled, lab)),
            'davies_bouldin': float(davies_bouldin_score(X_scaled, lab)),
            'calinski_harabasz': float(calinski_harabasz_score(X_scaled, lab)),
        })
    return pd.DataFrame(baris)


def titik_siku(ks, inertia) -> int:
    """Titik siku = jarak tegak lurus terjauh dari garis ujung ke ujung."""
    x = np.asarray(ks, dtype=float)
    y = np.asarray(inertia, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    x1, y1, x2, y2 = x_n[0], y_n[0], x_n[-1], y_n[-1]
    jarak = np.abs((y2 - y1) * x_n - (x2 - x1) * y_n + x2 * y1 - y2 * x1)
    jarak /= np.hypot(y2 - y1, x2 - x1)
    return int(x[int(np.argmax(jarak))])


def profil_relatif(df: pd.DataFrame, label) -> pd.DataFrame:
    """Simpangan rata-rata tiap klaster terhadap rata-rata seluruh data,
    dalam satuan simpangan baku. Baris = fitur, kolom = klaster."""
    d = df.copy()
    d['cluster'] = label
    rata = d.drop(columns='cluster').mean()
    sd = d.drop(columns='cluster').std().replace(0, np.nan)
    return ((d.groupby('cluster').mean() - rata) / sd).round(2).T


def nama_otomatis(profil: pd.DataFrame) -> dict[str, str]:
    """Dua fitur dengan simpangan terbesar secara mutlak, beserta arahnya.
    WAJIB diganti mahasiswa dengan nama yang bermakna."""
    nama = {}
    for c in profil.columns:
        kol = profil[c]
        kuat = kol.reindex(kol.abs().sort_values(ascending=False).index)[:2]
        nama[str(int(c))] = ', '.join(
            f'{n} {"tinggi" if v > 0 else "rendah"}' for n, v in kuat.items())
    return nama


def simpan_hasil(nomor: int, isi: dict) -> str:
    os.makedirs(FOLDER_HASIL, exist_ok=True)
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    with open(jalur, 'w', encoding='utf-8') as f:
        json.dump(isi, f, indent=2, ensure_ascii=False)
    return jalur


def baca_hasil(nomor: int) -> dict | None:
    jalur = f'{FOLDER_HASIL}/varian{nomor}.json'
    if not os.path.exists(jalur):
        return None
    with open(jalur, encoding='utf-8') as f:
        return json.load(f)

Overwriting alat.py


In [ ]:
def ambil_berkas():
    """Cadangan: unggah berkas manual bila folder Drive tak bisa diakses."""
    from google.colab import files
    for nama in files.upload():
        if nama.startswith('varian') and nama.endswith('.json'):
            tujuan = 'data/hasil_varian/' + nama
        elif nama.endswith('.pkl') or nama == 'segmen.json':
            tujuan = 'model/' + nama
        elif nama.endswith('.png'):
            tujuan = 'gambar/' + nama
        else:
            tujuan = 'data/' + nama
        shutil.move(nama, tujuan)
        print('->', tujuan)


def perlu(*jalur):
    """Memastikan luaran pertemuan sebelumnya sudah ada di folder Drive."""
    kurang = [j for j in jalur if not os.path.exists(j)]
    if not kurang:
        print('Lengkap:', ', '.join(jalur))
        return
    raise FileNotFoundError(
        'Belum ada di folder Drive: ' + ', '.join(kurang) + '. '
        'Jalankan notebook pertemuan sebelumnya sampai selesai. Bila folder '
        'kelompok tidak dapat diakses dari akun Google Anda, jalankan '
        'ambil_berkas() di sel baru untuk mengunggah berkasnya manual.')


print('Folder kerja:', os.getcwd())
print('perlu() dan ambil_berkas() siap dipakai.')

Folder kerja: /content/drive/.shortcut-targets-by-id/1kVQHXVw8QsRBgDDIfabawCG8Kc5AWXBQ/ProyekML_KlasteringKMeans
perlu() dan ambil_berkas() siap dipakai.


## 3.1 Mengambil berkas yang diperlukan

Sel ini memeriksa berkas yang Anda butuhkan di folder Drive kelompok. Bila ada yang belum dihasilkan, sel berhenti dengan pesan yang menyebut berkasnya. Aman dijalankan berulang kali.

In [ ]:
perlu('data/data_bersih.csv')

Lengkap: data/data_bersih.csv


## 3.2 Menjalankan varian 2

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

from alat import (SEED, baca_hasil, jalankan_k, muat_bersih, simpan_hasil,
                  skalakan)

print('=' * 70)
print('VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE')
print('=' * 70)

df = muat_bersih()
X, _ = skalakan(df)
print(f'Data bersih kelompok: {df.shape[0]} baris, {df.shape[1]} fitur\n')

tabel = jalankan_k(X)
print(tabel[['k', 'silhouette', 'davies_bouldin', 'inertia']]
      .round(4).to_string(index=False))

k_terbaik = int(tabel.loc[tabel['silhouette'].idxmax(), 'k'])
sil_terbaik = float(tabel['silhouette'].max())

print(f'\nSilhouette tertinggi : k = {k_terbaik}  ({sil_terbaik:.4f})')
if sil_terbaik >= 0.5:
    tafsir = 'pemisahan klaster jelas'
elif sil_terbaik >= 0.25:
    tafsir = 'struktur lemah tetapi masih dapat dipertanggungjawabkan'
else:
    tafsir = 'struktur klaster hampir tidak ada, pilihan fitur perlu ditinjau'
print(f'Tafsiran nilainya    : {tafsir}')

# --- Bandingkan dengan kesimpulan Varian 1 ---
v1 = baca_hasil(1)
print('\n' + '-' * 70)
if v1 is None:
    print('Hasil Varian 1 belum ada di sesi ini. Itu tidak masalah:')
    print('perbandingan ketiga kriteria disusun kelompok di Bagian 5.')
    k_elbow, beda, penjelasan = None, None, None
else:
    k_elbow = v1['k_terpilih']
    beda = k_elbow != k_terbaik
    print(f'Kesimpulan Varian 1 (Elbow)      : k = {k_elbow}')
    print(f'Kesimpulan Varian 2 (Silhouette) : k = {k_terbaik}')
    if not beda:
        penjelasan = ('Kedua varian menunjuk nilai k yang sama, sehingga '
                      'pilihan k berdiri di atas dua dasar yang saling '
                      'menguatkan.')
    else:
        sil_elbow = float(tabel.loc[tabel['k'] == k_elbow, 'silhouette'].iloc[0])
        selisih = sil_terbaik - sil_elbow
        penjelasan = (
            f'Kedua varian berbeda. Elbow mengukur seberapa rapat anggota '
            f'terhadap pusat klasternya sendiri, sedangkan Silhouette juga '
            f'memperhitungkan jarak ke klaster tetangga. Inertia selalu turun '
            f'saat k bertambah, sehingga titik sikunya bergantung pada bentuk '
            f'kurva, bukan pada mutu pemisahan. Pada k={k_elbow} Silhouette '
            f'bernilai {sil_elbow:.4f}, selisih {selisih:.4f} dari nilai '
            f'tertinggi di k={k_terbaik}.')
        if k_terbaik == 2:
            penjelasan += (' Perlu dicatat, Silhouette sering memuncak di k=2 '
                           'karena dua klaster paling mudah dipisahkan, '
                           'padahal dua segmen jarang berguna untuk '
                           'pengambilan keputusan.')
    print('\n' + penjelasan)
print('-' * 70)

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.plot(tabel['k'], tabel['silhouette'], marker='o', color='green')
plt.axvline(k_terbaik, ls='--', lw=1, color='grey')
if k_elbow:
    plt.axvline(k_elbow, ls=':', lw=1.5, color='tab:blue')
plt.xlabel('k'); plt.ylabel('Silhouette Score'); plt.title('Varian 2 - Silhouette')
# Panel kedua: diagram silhouette per sampel pada k terpilih.
# Tiap batang satu baris data; panjangnya nilai silhouette baris itu.
km = KMeans(n_clusters=k_terbaik, init='k-means++', n_init=20, random_state=SEED)
lab = km.fit_predict(X)
nilai = silhouette_samples(X, lab)

plt.subplot(1, 2, 2)
bawah = 10
for c in range(k_terbaik):
    v = np.sort(nilai[lab == c])
    atas = bawah + len(v)
    plt.fill_betweenx(np.arange(bawah, atas), 0, v, alpha=0.75)
    plt.text(-0.06, bawah + len(v) / 2, str(c), va='center', fontsize=9)
    bawah = atas + 10
plt.axvline(sil_terbaik, color='red', ls='--', lw=1,
            label=f'rata-rata {sil_terbaik:.3f}')
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Nilai silhouette tiap baris data'); plt.ylabel('Klaster')
plt.yticks([])
plt.title(f'Diagram silhouette pada k={k_terbaik}')
plt.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig('gambar/varian2_silhouette.png', dpi=110); plt.close()

negatif = int((nilai < 0).sum())
print(f'\nBaris dengan silhouette negatif: {negatif} dari {len(nilai)} '
      f'({negatif / len(nilai) * 100:.1f}%)')
print('Nilai negatif berarti baris itu lebih dekat ke klaster tetangga')
print('daripada ke klasternya sendiri. Sedikit baris seperti itu wajar;')
print('kalau jumlahnya besar, pemisahan klasternya memang lemah.')

jalur = simpan_hasil(2, {
    'varian': 2,
    'nama': 'Validasi k (Silhouette Score)',
    'yang_diubah': 'Jumlah klaster ditentukan dari Silhouette tertinggi',
    'fitur_dipakai': df.columns.tolist(),
    'k_terpilih': k_terbaik,
    'k_menurut_elbow': k_elbow,
    'berbeda_dari_elbow': beda,
    'penjelasan_perbedaan': penjelasan,
    'tafsiran': tafsir,
    'metrik_utama': {'silhouette': round(sil_terbaik, 4)},
    'baris_silhouette_negatif': negatif,
    'metrik_pendukung': {
        'inertia': round(float(tabel.loc[tabel['k'] == k_terbaik, 'inertia'].iloc[0]), 2),
        'davies_bouldin': round(float(tabel.loc[tabel['k'] == k_terbaik, 'davies_bouldin'].iloc[0]), 4),
    },
    'tabel_k': tabel[['k', 'inertia', 'silhouette', 'davies_bouldin']]
                .round(4).to_dict(orient='records'),
})
print(f'\nTersimpan: {jalur}  dan  gambar/varian2_silhouette.png')
print('Commit berkas ini dengan akun GitHub Anda sendiri.')

VARIAN 2 - VALIDASI k DENGAN SILHOUETTE SCORE
Data bersih kelompok: 1470 baris, 12 fitur

 k  silhouette  davies_bouldin    inertia
 2      0.2003          2.0103 14665.1090
 3      0.1274          2.2982 13535.9412
 4      0.1123          2.3654 12651.3222
 5      0.1126          2.2237 11924.1317
 6      0.1168          2.1732 11429.4561
 7      0.1114          2.0701 10972.7867
 8      0.0951          2.0666 10654.5781
 9      0.1030          2.1239 10385.1358
10      0.0933          2.1189 10134.1784

Silhouette tertinggi : k = 2  (0.2003)
Tafsiran nilainya    : struktur klaster hampir tidak ada, pilihan fitur perlu ditinjau

----------------------------------------------------------------------
Kesimpulan Varian 1 (Elbow)      : k = 5
Kesimpulan Varian 2 (Silhouette) : k = 2

Kedua varian berbeda. Elbow mengukur seberapa rapat anggota terhadap pusat klasternya sendiri, sedangkan Silhouette juga memperhitungkan jarak ke klaster tetangga. Inertia selalu turun saat k bertambah, sehin

## 3.3 Yang harus Anda tulis sendiri

Rubrik varian perorangan menilai tiga hal: **ketepatan implementasi**, **kebenaran evaluasi**, dan **kedalaman interpretasi**. Butir ketiga tidak dapat dihasilkan oleh kode, dan di situlah nilai paling banyak hilang.

1. Apakah kesimpulan Anda sama dengan Elbow di Bagian 2? Bila berbeda, jelaskan dengan kalimat Anda sendiri mengapa dua metrik dapat menunjuk k yang berbeda.
2. Bila Silhouette tertinggi ada di k=2, mengapa nilai itu tidak dipakai?
3. Apa arti angka Silhouette ini dalam konteks kasus kelompok Anda? Pakai patokan tafsiran di atas, jangan hanya menyebut angkanya.
##jawab
1. berbeda dari elbow, pad Elbow menunjukkan k=5, sedangkan di silhoute tertingginya k=2 (0,2003), karena inertia mengukur kerapatan anggota terhadap pusat klasternya sendiri dan selalu turun saat k bertambah, sehingga titik siku hanya menandai di mana penurunannya melandai. pada silhoute juga membandingkan dengan klaster tetangga. pada grafik silhoute hampir datar pada k=4 (0.1123), k=5 (0.1126), k=6 (0.1168), dan k=7 (0.1114), jadi k=5 (0,1126) tidak berbeda berarti dari tetangganya. hanya k=2 yang menonjol dengan selisih sebesar 0,0877 dari k=5.
2. nilai k=2 tidak saya pakai karena dua segmen karyawan terlalu kasar untuk dijadikan dasar rekomendasi tindakan. silhouette juga cenderung memuncak di k=2 karena dua kelompok besar paling mudah dipisahkan. lagi pula nilai tertingginya ada pada 0,2003, masih di bawah 0,25, jadi unggul di k=2 tidak berarti pemisahannya jelas.
3. nilai 0,2003 berada di bawah 0,25, berarti struktur klaster hampir tidak ada dan sebanyak 83 dari 1.470 baris (5,6%) bernilai negatif, artinya lebih dekat ke klaster tetangga daripada ke klasternya sendiri. Pada diagram k=2 juga ekor negatif hanya terlihat di klaster 0 yang lebih kecil, sedangkan klaster 1 batangnya semuanya di atas nol. davies bouldin pada tabel yang sama bernilai 2,01 sampai 2,37 semuanya di atas 1,5 sehingga dua metrik sepakat bahwa pemisahannya lemah. dugaan saya penyebabnya karena pilmiihan fitur dan ini akan diuji di bagian 5 (Perbandingan tiga kriteria dan pemilihan fitur)

## 3.4 Unduh, commit, dan teruskan

In [ ]:
# Unduh hasil Bagian 3, lalu commit ke repositori dengan akun GitHub
# ANDA SENDIRI. Riwayat commit adalah bukti penilaian perorangan.
from google.colab import files
files.download('data/hasil_varian/varian2.json')
files.download('gambar/varian2_silhouette.png')
print()
print('Langkah berikutnya:')
print('  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.')
print('  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,')
print('     lalu commit ke folder notebook/.')
print('  3. Teruskan `varian2.json` ke Anggota 3.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Langkah berikutnya:
  1. Commit kedua berkas di atas dengan akun GitHub Anda sendiri.
  2. File -> Download -> Download .ipynb, beri nama 03_varian_2.ipynb,
     lalu commit ke folder notebook/.
  3. Teruskan `varian2.json` ke Anggota 3.


## Checklist Bagian 3

- [ ] `varian2.json` ter-commit di `data/hasil_varian/`
- [ ] `varian2_silhouette.png` ter-commit di `gambar/`
- [ ] Notebook ini diunduh, diberi nama `03_varian_2.ipynb`, dan ter-commit di `notebook/`
- [ ] Commit dilakukan dengan akun GitHub Anggota 2 sendiri
- [ ] Perbandingan dengan kesimpulan Elbow sudah ditulis
- [ ] `varian2.json` sudah dikirim ke Anggota 3